# 03 — Feature Engineering

**Goal:** Build the model-ready feature matrix: technical indicators from OHLCV, lagged LLM sentiment signals, and a binary 5-day directional target. Also inspect class balance.


## 3.1 Imports & load


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'Data' / 'cryptonews.csv').exists() or (ROOT / 'notebooks').exists():
        break
    if ROOT == ROOT.parent:
        break
    ROOT = ROOT.parent
INTERIM = ROOT / 'notebooks' / 'interim'

df = pd.read_parquet(INTERIM / 'merged_with_llm_sentiment.parquet')
df = df.sort_values('date').reset_index(drop=True)
print(f'{len(df):,} rows | {df.date.min()} → {df.date.max()}')
df.head(3)

## 3.2 Technical indicators


In [ ]:
def rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    rs = gain / (loss + 1e-12)
    return 100 - (100 / (1 + rs))

def macd(close: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9):
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line, macd_line - signal_line

def bollinger(close: pd.Series, window: int = 20, k: float = 2.0):
    mid = close.rolling(window).mean()
    std = close.rolling(window).std()
    return mid, mid + k * std, mid - k * std

# Apply
df['ret_1d'] = np.log(df['close'] / df['close'].shift(1))
df['ret_3d'] = np.log(df['close'] / df['close'].shift(3))
df['ret_7d'] = np.log(df['close'] / df['close'].shift(7))
df['vol_7d'] = df['ret_1d'].rolling(7).std()
df['vol_21d'] = df['ret_1d'].rolling(21).std()
df['rsi_14'] = rsi(df['close'], 14)
macd_line, sig_line, hist = macd(df['close'])
df['macd_line'] = macd_line
df['macd_hist'] = hist
bb_mid, bb_up, bb_low = bollinger(df['close'])
df['bb_pct_b'] = (df['close'] - bb_low) / (bb_up - bb_low + 1e-12)
df['bb_width'] = (bb_up - bb_low) / (bb_mid + 1e-12)

df.tail(3)

## 3.3 Lagged LLM sentiment features


In [ ]:
for lag in [1, 2, 3, 5]:
    df[f'llm_sent_lag{lag}'] = df['llm_sentiment_mean'].shift(lag)
    df[f'llm_pos_share_lag{lag}'] = df['llm_pos_share'].shift(lag)

# 3-day rolling mean of LLM sentiment (momentum)
df['llm_sent_3d_ma'] = df['llm_sentiment_mean'].rolling(3).mean().shift(1)
df['llm_sent_5d_ma'] = df['llm_sentiment_mean'].rolling(5).mean().shift(1)

df[['date', 'close', 'llm_sentiment_mean', 'llm_sent_lag1', 'llm_sent_3d_ma']].tail(5)

## 3.4 5-day directional target
Binary classification: `1` if BTC closes higher 5 days from now, else `0`. This is the variable the LSTM learns to predict.


In [ ]:
HORIZON = 5
df['forward_ret_5d'] = df['close'].shift(-HORIZON) / df['close'] - 1
df['target_up_5d'] = (df['forward_ret_5d'] > 0).astype(int)

df = df.dropna(subset=['target_up_5d']).reset_index(drop=True)

print('Class balance (1 = up, 0 = down):')
print(df['target_up_5d'].value_counts(normalize=True).round(3))
ax = df['target_up_5d'].value_counts().plot(kind='bar', title='5-day directional target distribution')
ax.set_xticklabels(['Down (0)', 'Up (1)'], rotation=0)
plt.tight_layout(); plt.show()

## 3.5 Train / validation / test split
Time-ordered split — never shuffle time series. 70% train, 15% validation, 15% test.


In [ ]:
n = len(df)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)
n_test  = n - n_train - n_val

train = df.iloc[:n_train].copy()
val   = df.iloc[n_train:n_train+n_val].copy()
test  = df.iloc[n_train+n_val:].copy()

print(f'Train: {len(train):>4}  {train.date.min()} → {train.date.max()}')
print(f'Val  : {len(val):>4}  {val.date.min()} → {val.date.max()}')
print(f'Test : {len(test):>4}  {test.date.min()} → {test.date.max()}')

## 3.6 Feature matrix & scaling


In [ ]:
FEATURE_COLS = [
    'ret_1d', 'ret_3d', 'ret_7d', 'vol_7d', 'vol_21d',
    'rsi_14', 'macd_line', 'macd_hist', 'bb_pct_b', 'bb_width',
    'llm_sent_lag1', 'llm_sent_lag2', 'llm_sent_lag3', 'llm_sent_lag5',
    'llm_pos_share_lag1', 'llm_pos_share_lag3', 'llm_pos_share_lag5',
    'llm_sent_3d_ma', 'llm_sent_5d_ma',
    'news_count', 'mean_polarity', 'neg_share', 'pos_share',
]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_x = scaler.fit_transform(train[FEATURE_COLS].fillna(0))
val_x   = scaler.transform(val[FEATURE_COLS].fillna(0))
test_x  = scaler.transform(test[FEATURE_COLS].fillna(0))

train_y = train['target_up_5d'].values
val_y   = val['target_up_5d'].values
test_y  = test['target_up_5d'].values

# Reshape for LSTM: (samples, timesteps, features)
TIMESTEPS = 1
train_x = train_x.reshape(-1, TIMESTEPS, len(FEATURE_COLS))
val_x   = val_x.reshape(-1, TIMESTEPS, len(FEATURE_COLS))
test_x  = test_x.reshape(-1, TIMESTEPS, len(FEATURE_COLS))

print(f'train_x: {train_x.shape}  train_y: {train_y.shape}')
print(f'val_x  : {val_x.shape}    val_y  : {val_y.shape}')
print(f'test_x : {test_x.shape}   test_y : {test_y.shape}')
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

## 3.7 Persist for Notebook 04


In [ ]:
import pickle

bundle = {
    'train_x': train_x, 'train_y': train_y,
    'val_x': val_x, 'val_y': val_y,
    'test_x': test_x, 'test_y': test_y,
    'feature_cols': FEATURE_COLS,
    'scaler': scaler,
    'train_dates': train['date'].values,
    'val_dates': val['date'].values,
    'test_dates': test['date'].values,
    'test_close': test['close'].values,  # for backtesting
    'test_forward_ret_5d': test['forward_ret_5d'].values,
}
out_path = INTERIM / 'features_for_lstm.pkl'
with out_path.open('wb') as f:
    pickle.dump(bundle, f)
print(f'Wrote {out_path}  ({out_path.stat().st_size / 1024:.1f} KB)')

## 3.8 Summary
- Built 23 features: technical indicators (RSI, MACD, Bollinger %B/width, vol, returns) + lagged LLM sentiment + rolling sentiment momentum.
- Created 5-day directional target; observed class imbalance that Notebook 04 will handle via `class_weight`.
- Time-ordered 70/15/15 split with StandardScaler fitted only on train.
- Persisted `(samples, 1, 23)` tensors + scaler + test-set BTC closes (for backtesting) to `notebooks/interim/features_for_lstm.pkl`.
